# iri_client Demo

We'll start off by importing some libraries we'll be using in the demo

In [ ]:
# # If using jupyter.nersc.gov and you haven't already, install the client
# !pip install nersc-iri-client
#
# The IRI API uses a statically-issued bearer token rather than the
# Superfacility client_id/client_secret key pair.  We'll cover how to
# make it available in Exercise 2.

In [ ]:
from iri_client import (
    Client, JobState, FilesystemResourceName, ComputeResourceName,
)
from iri_client.exceptions import AuthError
from iri_client._models import JobSpec

from datetime import datetime
from pathlib import Path
import os

***
# Exercise 1 - Un-Authenticated Client
## Check the IRI service status
### These can all be done without a bearer token
***
Before we start any computing, let's check the status of the NERSC resources served by IRI.  The SFAPI client hung these operations off `client.resources`. IRI models every system as a *resource* with an id, name, description, and a current `Status` (`up`, `down`, `degraded`, or `unknown`).

In [ ]:
with Client() as client:
    nersc_status = client.status.statuses()

# `statuses()` returns a name -> Status map for every resource
for name, status in nersc_status.items():
    print(f"{name: <24}| {status}")

We can look at the underlying resources (with their descriptions) under `status.resources()`

In [ ]:
with Client() as client:
    resources = client.status.resources()

# For each resource print its name, description, and current status
for resource in resources:
    name = resource.name or resource.id
    print(f"{name: <24}| {resource.description or '': <26}| {resource.current_status}")

The status router also exposes `incidents` (planned and unplanned outages, the IRI analog of the SFAPI `outages` call)

In [ ]:
with Client() as client:
    incidents = client.status.incidents()

print(f"{len(incidents)} incidents on record:")
for incident in incidents:
    name = incident.name or incident.description or incident.id
    print(f"  {incident.start} | {incident.type} | {incident.status} | {name}")

We can check any incidents affecting a resource this month

In [ ]:
today = datetime.now().date()
for incident in incidents:
    if not incident.end or incident.end.date() >= today:
        print(incident)
        print("---")

And `status.events()` records each status *change* of a resource

In [ ]:
with Client() as client:
    events = client.status.events()

for event in events[:5]:
    print(f"{event.occurred_at} | {event.status} | {event.name or event.resource_uri}")

What if we try to get more information from the API with our un-authenticated client?

In [ ]:
with Client() as client:
    # We get an error!
    try:
        client.account.projects()
    except AuthError as e:
        print(type(e).__name__ + ':', e)

# 

<br>
<br>
<br>
<br>
# Exercise 2 - Authenticated Client
## Get a token, then user and project information
***
The Superfacility API authenticates with a private-key JWT client-credentials flow (the `client_id`/`client_secret` files).  IRI instead expects a statically-issued bearer token. `iri_client` resolves it by priority:

    1. the `access_token=` constructor argument
    2. the `IRI_API_TOKEN` environment variable
    3. the file `~/.ssh/nersc-token`

In [ ]:
# Let's make sure the client can find a token.
token_path = Path.home() / ".ssh" / "nersc-token"
print(f"token file exists: {token_path.exists()}")

# If your token is not in the file, set it explicitly here:
# os.environ["IRI_API_TOKEN"] = "<paste the token from iri.nersc.gov>"

# Or pass it directly to any client you create:
# Client(access_token="<paste the token>")

### We'll use this token through the rest of our tutorial.
Let's make sure we're authenticated and pull some information from the API

In [ ]:
# Create a client
with Client() as client:
    # Get the projects the API thinks I belong to
    projects = client.account.projects()

### All data returned from the API is a model object
Objects are specific to the type of information returned from the API and expose their attributes directly, or can be serialized as dictionaries or JSON with pydantic.

In [ ]:
# We get a list of project objects
print(type(projects[0]).__name__)

# Each project has attributes such as name and id
print(projects[0].name)
print(projects[0].id)

# Or we can return json which is handy for other libraries
print(projects[0].model_dump_json())

### We can also get the *allocations* and *usage* for each project
A project carries layered allocations; `account.usage(project_id)` aggregates every allocation entry (granted vs. consumed, in units such as `node_hours`).

In [ ]:
# For each of my projects print the usage information
print("Project              |              Allocation (unit: hours)")
for project in projects:
    with Client() as client:
        usage = client.account.usage(project.id)
    for entry in usage:
        print("=" * 60)
        print(f"{project.name: <18} | {entry.usage:>12.2f} of {entry.allocation:>12.2f} {entry.unit}")

# 

<br>
<br>
# Exercise 3 - Filesystem interactions and small file upload/download
## Interact with the IRI filesystem tasks
***
Now that we're authenticated we can use the IRI *filesystem* resources. 
IRI exposes several filesystems. Instead of remembering string identifiers you can pick one with the `FilesystemResourceName` enum (`scratch`, `homes`, `common`, `cfs`, `dtns`, `jobs`, `compute`, `login`).

Every IRI filesystem operation is **asynchronous**: the API accepts the request and returns a task id immediately, and the client polls the task until it finishes. The `submit_*` helpers expose the raw task; the plain methods (as used below) submit *and* block for you.

Let's make some useful variables for your home and scratch directories. They are based on your username:

* `/global/homes/username_first_letter/username`
* `/pscratch/sd/username_first_letter/username`

In [ ]:
username = os.environ["USER"]
home = f"/global/homes/{username[0]}/{username}"
scratch = f"/pscratch/sd/{username[0]}/{username}"

scratch

Lets make a directory in `$SCRATCH` to put things from the API training demos. Filesystem operations go through the `scratch` filesystem resource, which we select with `FilesystemResourceName.scratch` (a plain `"scratch"` string works too).

In [ ]:
with Client() as client:
    fs = client.filesystem(FilesystemResourceName.scratch)
    # Create our demo directory (tasks!)
    fs.mkdir(f"{scratch}/iri-demo", parent=True)
    entries = fs.list(f"{scratch}/iri-demo")

# Check that the directory is there and empty
print(entries)

Next let's upload a small file to that directory and make sure it's there.

In [ ]:
# What we want in our file
file_contents = "hello world!\n"
my_input_file = Path("/tmp/iri-demo-hello.txt")
my_input_file.write_text(file_contents)


In [ ]:
with Client() as client:
    fs = client.filesystem(FilesystemResourceName.scratch)
    remote = f"{scratch}/iri-demo/hello.txt"
    before = len(fs.list(f"{scratch}/iri-demo"))
    print(f"There are {before} files in the directory")
    fs.upload(str(my_input_file), remote)
    after = len(fs.list(f"{scratch}/iri-demo"))
    print(f"Now there are {after} files in the directory")

Let's verify that we put some text in that file, using `view`, `head`, `tail`, `checksum`, and `download`. Each of these unwraps the useful payload for you — you get the file text (or hash) directly, not a metadata wrapper.

In [ ]:
with Client() as client:
    fs = client.filesystem(FilesystemResourceName.scratch)
    print("view:", fs.view(remote))
    print("head:", fs.head(remote, lines=1))
    print("tail:", fs.tail(remote, lines=1))
    print("checksum:", fs.checksum(remote))
    print("download:", fs.download(remote))

# 

<br>
<br>
# Exercise 4 - Submitting jobs to the compute resource
## Getting job information and submitting work
***
Now we'll connect to the IRI *compute* resource and interact with the backend scheduler to get information about jobs as well as submit new work. Compute resources are selected with the `ComputeResourceName` enum (`jobs`, `compute`, or `perlmutter`); the default binding, `"jobs"`, is the resource the IRI deployment expects job submission against, and `perlmutter` is an alias for that same jobs group. So `client.compute(ComputeResourceName.perlmutter)`, `client.compute(ComputeResourceName.jobs)`, or `client.compute("jobs")` all work.

Let's check how many jobs are currently running for our user.

In [ ]:
with Client() as client:
    compute = client.compute(ComputeResourceName.perlmutter)
    current_jobs = compute.list(filters={"user": username})

running_jobs = [job for job in current_jobs if job.status.state == JobState.active]
print(f"{len(running_jobs)} running job(s) out of {len(current_jobs)} in the queue view")

Now let's run a job and see how to interact with it through the API. We'll start with a very simple python code to generate random numbers from a normal distribution.

IRI jobs are described by a `JobSpec`, not a shell job script. The `attributes` carry the scheduler options such as the queue/partition and the account to charge.

In [ ]:
# Fill in the account (project) to charge for this job
N = 10000
account = "m3792"       # <- your IRI/Slurm account
queue_name = "debug"    # <- queue or partition to submit to

# A one-liner that prints N random normal variates
code = ";".join([
    "import random, math",
    f"numbers=[random.gauss(0, 1) for _ in range({N})]",
    "[print(n) for n in numbers]",
])

jobspec = JobSpec(
    name="iri-demo",
    executable="/usr/bin/python3",
    arguments=["-c", code],
    directory=scratch,
    stdout_path=f"{scratch}/iri-demo/iri-demo.out",
    stderr_path=f"{scratch}/iri-demo/iri-demo.err",
    inherit_environment=False,
    attributes={
        "queue_name": queue_name,
        "account": account,
        "duration": 120,
    },
    resources={"node_count": 1, "processes_per_node": 1},
)

print(jobspec.model_dump(exclude_none=True))

In [ ]:
with Client() as client:
    compute = client.compute(ComputeResourceName.perlmutter)
    job = compute.submit(jobspec)
    # Let's save the job id to use later
    job_id = job.id
    print(f"Started {job_id}!")

In [ ]:
with Client() as client:
    compute = client.compute(ComputeResourceName.perlmutter)
    # Fetch the submitted job back by id (the queue view can lag)
    job = compute.job(job_id)
    print(job.status.state)

You can also have the client wait for the job to complete. It will poll the API until the job reaches a terminal state.

In [ ]:
with Client() as client:
    compute = client.compute(ComputeResourceName.perlmutter)
    job = compute.job(job_id)
    print(f"Waiting for {job.id} to complete...")
    job.wait()
    print(job.status.state, "exit code:", job.status.exit_code)

Now the job is done let's download the results file and read it into a variable.

In [ ]:
with Client() as client:
    fs = client.filesystem(FilesystemResourceName.scratch)
    contents = fs.download(f"{scratch}/iri-demo/iri-demo.out")

output_numbers = contents

In [ ]:
# How many numbers did we get back?
print(f"got {len(output_numbers.splitlines())} numbers")

Then we can look at our output results from the job we ran.

In [ ]:
import matplotlib.pyplot as plt

# Convert text numbers into a list we can use with matplotlib
numbers = list(map(float, output_numbers.splitlines()))

plt.hist(numbers, bins=100)
plt.show()

Clean up the demo directory (optional)

In [ ]:
# Clean up the directory we created in Exercise 3
with Client() as client:
    client.filesystem(FilesystemResourceName.scratch).rm(f"{scratch}/iri-demo")
    my_input_file.unlink(missing_ok=True)
    print("cleaned up")